In [0]:
import importlib
from config.config import *

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta").load(
    f"{bronze_path}/bronze_world_bank_internatinal_arrival_raw"
)
display(df.head(10))

In [0]:
header_row = df.first()

new_columns = [
    str(header_row[c]).strip().lower().replace(" ", "_")
    for c in df.columns
]

df = df.filter(
    col("_c0") != header_row["_c0"]
)

df = df.toDF(*new_columns)

In [0]:
string_columns = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

for c in string_columns:
    df = df.withColumn(c, trim(col(c)))

In [0]:
year_columns = [
    c for c in df.columns if c.isdigit()
]

year_array = array(*[
    struct(
        lit(int(year)).alias("year"),
        col(year).alias("raw_arrivals")
    )
    for year in year_columns
])

df = df.withColumn("year_data", explode(year_array))


In [0]:
df = df.select(
    "country_name",
    "country_code",
    "indicator_name",
    "indicator_code",
    col("year_data.year").alias("year"),
    col("year_data.raw_arrivals").alias("raw_arrivals")
)


In [0]:
df = df.withColumn("year", col("year").cast("int"))

df = df.withColumn(
    "arrivals",
    expr("try_cast(raw_arrivals AS DOUBLE)")
)


In [0]:
aggregate_codes = ["AFE", "AFW", "ARB"]

df = df.withColumn(
    "geography_type",
    when(
        col("country_code").isin(*aggregate_codes),
        "regional_aggregate"
    ).otherwise("country")
)

In [0]:
silver_df = df.select(
    "country_name",
    "country_code",
    "indicator_name",
    "indicator_code",
    "geography_type",
    "year",
    "arrivals"
)

In [0]:
output_path = f"{silver_path}/world_bank_international_arrivals"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_path)

In [0]:
spark.read.format("delta").load(output_path).display()